In [0]:
from pyspark.sql import functions as F

from pyspark.sql.window import Window

from delta.tables import DeltaTable

# ---------------- CONFIG ----------------

BRONZE_SLA = "workspace.slainte_bronze.sla_groups_bronze"

GOLD_DB    = "workspace.slainte_gold"

DIM_SLA    = f"{GOLD_DB}.dim_sla"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- BUILD BASE ----------------

df_dim_sla = (

    spark.table(BRONZE_SLA)

    .select(

        F.col("user_id").alias("source_user_id"),

        F.col("project_key").alias("project"),

        F.col("group_name").alias("priority_code"),

        F.col("response_time_minutes").cast("int"),

        F.col("resolution_time_hours").cast("int"),

        F.col("team"),

        F.col("country"),

        F.col("calendar_type"),

        F.col("working_days"),

        F.col("work_start_time"),

        F.col("work_end_time"),

        F.col("timezone")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("project").isNotNull() &

        F.col("priority_code").isNotNull() &

        F.col("response_time_minutes").isNotNull()

    )

    .distinct()

)

# ---------------- PRIORITY RANK ----------------

window_spec = Window.partitionBy("source_user_id", "project").orderBy(

    F.col("response_time_minutes").asc(),

    F.col("resolution_time_hours").asc()

)

df_dim_sla = df_dim_sla.withColumn(

    "priority_label",

    F.row_number().over(window_spec)

)

# ---------------- MERGE KEY ----------------

merge_condition = """

t.source_user_id = s.source_user_id AND

t.project = s.project AND

t.priority_label = s.priority_label

"""

# ---------------- INITIAL LOAD ----------------

if not spark.catalog.tableExists(DIM_SLA):

    df_dim_sla = df_dim_sla.withColumn(

        "sla_id", F.monotonically_increasing_id() + 1

    )

    df_dim_sla.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(DIM_SLA)

    print("✅ dim_sla created (initial load)")

else:

    delta_table = DeltaTable.forName(spark, DIM_SLA)

    (

        delta_table.alias("t")

        .merge(

            df_dim_sla.alias("s"),

            merge_condition

        )

        .whenMatchedUpdate(set={

            "priority_code": "s.priority_code",

            "response_time_minutes": "s.response_time_minutes",

            "resolution_time_hours": "s.resolution_time_hours",

            "team": "s.team",

            "country": "s.country",

            "calendar_type": "s.calendar_type",

            "working_days": "s.working_days",

            "work_start_time": "s.work_start_time",

            "work_end_time": "s.work_end_time",

            "timezone": "s.timezone"

        })

        .whenNotMatchedInsert(values={

            "sla_id": "monotonically_increasing_id()",

            "source_user_id": "s.source_user_id",

            "project": "s.project",

            "priority_code": "s.priority_code",

            "priority_label": "s.priority_label",

            "response_time_minutes": "s.response_time_minutes",

            "resolution_time_hours": "s.resolution_time_hours",

            "team": "s.team",

            "country": "s.country",

            "calendar_type": "s.calendar_type",

            "working_days": "s.working_days",

            "work_start_time": "s.work_start_time",

            "work_end_time": "s.work_end_time",

            "timezone": "s.timezone"

        })

        .execute()

    )

    print("✅ dim_sla merged successfully")

# ---------------- DISPLAY ----------------

display(

    spark.table(DIM_SLA)

    .orderBy("source_user_id", "project", "priority_label")

)
 